In [ ]:
# === Mount Google Drive and install dependencies ===
# Drive must be mounted first so pip can find requirements.txt.
# This cell is safe to re-run if Drive is already mounted.
from google.colab import drive
drive.mount("/content/drive")
!pip install -r /content/drive/MyDrive/iris/requirements.txt -q

# 01 â€” Data Exploration & J1 Separability Analysis

**Purpose:** Build the initial dataset (500 normal + 500 injection prompts),
extract residual stream activations at all 12 GPT-2 layers, and compute
the separability metrics for Experiment J1.

**Prerequisites:**
- Google Colab with GPU runtime (T4 is fine)
- Project mounted at `/content/drive/MyDrive/iris/`
- `pip install -r requirements.txt` already run

**J1 Pass Criterion:** At least one layer shows silhouette score > 0.1
or visible clustering in the t-SNE plot.

**Outputs:**
- `data/processed/iris_dataset_j1.json` â€” the 1000-example dataset
- `results/figures/j1_separability_by_layer.png` â€” bar chart of metrics
- `results/figures/j1_tsne_layer_*.png` â€” t-SNE plots for best layers
- `results/metrics/j1_separability.json` â€” raw metric values

In [ ]:
# === Setup: path configuration and imports ===
# In Colab, the project lives on Google Drive. We add the project root
# to sys.path so that `from src.xxx import yyy` works from notebooks.
import sys
import os

# Detect whether we're in Colab or running locally
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Mount Google Drive if not already mounted
    PROJECT_ROOT = "/content/drive/MyDrive/iris"
else:
    # Local development â€” notebook is in iris/notebooks/
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add project root to Python path so we can import from src/
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)  # so relative paths in src/ modules resolve correctly

print(f"Project root: {PROJECT_ROOT}")
print(f"Working directory: {os.getcwd()}")

In [ ]:
# === Reproducibility: seed everything before any randomness occurs ===
from src.utils.helpers import set_seed, get_device
set_seed(42)
device = get_device()

## Step 1: Build the Dataset

We fetch 500 normal prompts from Stanford Alpaca and 500 injection prompts
from deepset/prompt-injections. These are combined into a single dataset
with a consistent schema.

**Why these numbers?** 1000 total examples is enough for J1 (a go/no-go
feasibility check). The full dataset for training will be larger (5000+
per class), but for junction experiments we want fast iteration.

In [ ]:
# === Fetch prompts from public HuggingFace datasets ===
from src.data.sources import fetch_all

raw_examples = fetch_all(n_normal=500, n_injection=500, seed=42)

In [ ]:
# === Wrap in our dataset class and inspect ===
from src.data.dataset import IrisDataset

dataset = IrisDataset(raw_examples)
dataset.summary()

In [ ]:
# === Look at a few examples from each class ===
# Sanity check: make sure normal prompts look benign and injection
# prompts look adversarial. If these look wrong, the entire pipeline
# will produce meaningless results.
print("=== NORMAL EXAMPLES ===")
for ex in dataset.examples[:3]:
    print(f"  [{ex['source']}] {ex['text'][:100]}...")
    print()

print("=== INJECTION EXAMPLES ===")
for ex in dataset.examples[-3:]:
    print(f"  [{ex['source']}] {ex['text'][:100]}...")
    print()

In [ ]:
# === Add token counts and inspect the distribution ===
# Token counts tell us whether the two classes have very different
# lengths. If injections are systematically longer or shorter than
# normal prompts, the model might use length as a trivial cue rather
# than semantic content â€” which would be a confound.
from src.data.preprocessing import add_token_counts

formatted_prompts = dataset.format_prompts()
add_token_counts(dataset.examples, formatted_prompts)

dataset.summary()

In [ ]:
# === Visualize token count distributions by class ===
# This plot checks for the length confound described above.
import matplotlib.pyplot as plt
import numpy as np

normal_tokens = [ex["token_count"] for ex in dataset.examples if ex["label"] == 0]
inject_tokens = [ex["token_count"] for ex in dataset.examples if ex["label"] == 1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(normal_tokens, bins=30, alpha=0.6, label="Normal", color="#4C72B0")
ax.hist(inject_tokens, bins=30, alpha=0.6, label="Injection", color="#DD8452")
ax.set_xlabel("Token Count (with system prompt)")
ax.set_ylabel("Frequency")
ax.set_title("Token Count Distribution by Class")
ax.legend()
plt.tight_layout()
fig.savefig("results/figures/j1_token_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"Normal:    mean={np.mean(normal_tokens):.0f}, median={np.median(normal_tokens):.0f}")
print(f"Injection: mean={np.mean(inject_tokens):.0f}, median={np.median(inject_tokens):.0f}")

In [ ]:
# === Save the dataset ===
# We save before the expensive activation extraction step so that
# if the GPU session crashes, we don't need to re-download data.
sha256 = dataset.save("data/processed/iris_dataset_j1.json")
print(f"\nRecord this hash for reproducibility: {sha256}")

## Step 2: Extract Activations at All 12 Layers

We load GPT-2 Small via TransformerLens and extract the residual stream
activation at the last real token position for every prompt at every layer.

This produces 12 arrays, each of shape (1000, 768) â€” one 768-dimensional
vector per prompt per layer. These are the representations we'll analyze
for separability.

**Why last-token activations?** GPT-2 is autoregressive â€” the last token
has attended to the entire input, so its residual stream captures the
model's full "understanding" of the prompt. See Design Document Â§5.4.

In [ ]:
# === Load GPT-2 Small ===
from src.model.transformer import load_model

model = load_model(device=device)

In [ ]:
# === Tokenize all formatted prompts ===
# We tokenize the system-prompt-wrapped versions, because that's
# what the model will process. The system prompt framing is critical:
# it establishes the trust boundary that injections try to cross.
from src.data.preprocessing import tokenize_prompts

tokenized = tokenize_prompts(formatted_prompts, max_length=128)
print(f"input_ids shape:     {tokenized['input_ids'].shape}")
print(f"attention_mask shape: {tokenized['attention_mask'].shape}")

# Sanity check: how many prompts were truncated?
# If most prompts are truncated, we might be losing important
# information from the end of longer prompts.
n_truncated = (tokenized['attention_mask'].sum(dim=1) == 128).sum().item()
print(f"\nPrompts truncated to 128 tokens: {n_truncated}/{len(formatted_prompts)}")

In [ ]:
# === Extract residual stream activations at all 12 layers ===
# This is the most GPU-intensive step. On a T4, it takes ~2-3 minutes
# for 1000 prompts at 128 tokens with batch_size=32.
from src.model.transformer import extract_activations

activations = extract_activations(
    model=model,
    input_ids=tokenized["input_ids"],
    attention_mask=tokenized["attention_mask"],
    layers=list(range(12)),  # all 12 layers
    batch_size=32,
)

In [ ]:
# === Free GPU memory â€” we don't need the transformer anymore ===
# The activations are numpy arrays on CPU; the model is on GPU.
# Deleting it frees ~500MB of GPU memory for the analysis steps.
import torch
del model
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print("Model unloaded from GPU.")

In [ ]:
# === Save activations to disk ===
# This lets us skip re-extraction if the notebook kernel restarts.
# We save as .npz (compressed numpy) â€” 12 arrays of (1000, 768) float32
# is about 35 MB uncompressed, ~20 MB compressed.
save_dict = {f"layer_{i}": activations[i] for i in range(12)}
save_dict["labels"] = np.array(dataset.labels)
np.savez_compressed("checkpoints/j1_activations.npz", **save_dict)
print("Activations saved to checkpoints/j1_activations.npz")

## Step 3: J1 â€” Compute Separability Metrics

Now we answer the key question: **do normal and injection prompts
produce distinguishable activations at any layer?**

We compute two complementary metrics at each layer:
- **Silhouette score**: measures cluster quality (pass threshold: > 0.1)
- **Cohen's d**: measures standardized distance between class centroids

In [ ]:
# === (Optional) Reload activations if kernel restarted ===
# Uncomment these lines if you need to resume from a saved checkpoint
# without re-running the extraction.
#
# data = np.load("checkpoints/j1_activations.npz")
# activations = {i: data[f"layer_{i}"] for i in range(12)}
# labels = data["labels"].tolist()
# dataset = IrisDataset.load("data/processed/iris_dataset_j1.json")

In [ ]:
# === Compute separability at every layer ===
from src.analysis.separability import compute_all_layers

labels = dataset.labels
metrics = compute_all_layers(activations, labels)

In [ ]:
# === Check J1 pass criterion ===
# The Design Document says: "At least one layer shows visible clustering
# or silhouette score > 0.1."
best_layer = max(metrics, key=lambda l: metrics[l]["silhouette"])
best_sil = metrics[best_layer]["silhouette"]
best_d = metrics[best_layer]["cohens_d"]

print(f"\n{'='*50}")
print(f"BEST LAYER: {best_layer}")
print(f"  Silhouette score: {best_sil:.4f}")
print(f"  Cohen's d:        {best_d:.4f}")
print(f"{'='*50}")

if best_sil > 0.1:
    print("\nâœ“ J1 PASSED â€” silhouette > 0.1 at layer", best_layer)
    print("  Proceed with J2 (SAE sanity check).")
else:
    print("\nâœ— J1 FAILED â€” no layer exceeds silhouette threshold of 0.1")
    print("  Check Cohen's d and t-SNE visualizations before deciding.")
    print("  If Cohen's d > 0.5 or t-SNE shows any clustering, J1 may")
    print("  still pass on the 'visible clustering' criterion.")

In [ ]:
# === Plot separability metrics across layers ===
from src.analysis.separability import plot_separability_by_layer

plot_separability_by_layer(
    metrics,
    save_path="results/figures/j1_separability_by_layer.png",
)

In [ ]:
# === t-SNE visualization of the best layer ===
# This is the qualitative check: even if the silhouette score is low,
# visible clusters in t-SNE would pass J1's alternative criterion.
from src.analysis.separability import plot_activation_tsne

plot_activation_tsne(
    activations[best_layer],
    labels,
    layer=best_layer,
    save_path=f"results/figures/j1_tsne_layer_{best_layer}.png",
)

In [ ]:
# === t-SNE for the top 3 layers (for comparison) ===
# Looking at multiple layers helps us understand whether separability
# emerges gradually (increases with depth) or appears at specific layers.
top_3_layers = sorted(metrics, key=lambda l: metrics[l]["silhouette"],
                      reverse=True)[:3]

for layer in top_3_layers:
    if layer == best_layer:
        continue  # already plotted above
    print(f"\n--- Layer {layer} (silhouette={metrics[layer]['silhouette']:.4f}) ---")
    plot_activation_tsne(
        activations[layer],
        labels,
        layer=layer,
        save_path=f"results/figures/j1_tsne_layer_{layer}.png",
    )

In [ ]:
# === Save metrics to JSON for the project report ===
import json

# Convert keys to strings for JSON serialization
# (JSON keys must be strings, but our layer indices are ints)
metrics_serializable = {
    str(layer): vals for layer, vals in metrics.items()
}
metrics_serializable["best_layer"] = best_layer
metrics_serializable["j1_passed"] = best_sil > 0.1

metrics_path = "results/metrics/j1_separability.json"
os.makedirs(os.path.dirname(metrics_path), exist_ok=True)
with open(metrics_path, "w") as f:
    json.dump(metrics_serializable, f, indent=2)
print(f"Metrics saved to {metrics_path}")

## Summary

This notebook completed:
1. **Dataset construction** â€” 500 normal (Alpaca) + 500 injection (deepset) prompts
2. **Activation extraction** â€” residual stream at all 12 GPT-2 layers, last-token position
3. **J1 separability analysis** â€” silhouette scores, Cohen's d, and t-SNE visualizations

**Next steps:**
- If J1 passed â†’ proceed to `02_classical_baseline.ipynb` and then J2 (SAE sanity check)
- If J1 failed â†’ review t-SNE plots for partial clustering; consider pivoting to Path A